# SAE Adversarial Robustness — Master Experiment v3

**GPT-2 Small / Gemma 2 2B · Single-file Colab notebook**

Empirical investigation of whether Sparse Autoencoder (SAE) interpretations of
language model internals are robust to adversarial input perturbations. All six
experiments run end-to-end from this single notebook, with per-checkpoint Drive
saves so a Colab disconnect never loses results.

---

## Table of Contents

| # | Section | Research Q |
|---|---------|-----------|
| 1 | Smart Installer | — |
| 2 | HuggingFace Authentication | — |
| 3 | Model Configuration ← **switch model here** | — |
| 4 | Drive Mount + Checkpoint System | — |
| 5 | Model, SAE & Dataset Loading | — |
| 6 | **Exp 0** — Baseline Stats | — |
| 7 | **Exp 1** — Feature Stability under PGD | Q1, Q2 |
| 8 | **Exp 2** — Output-Preserving Attack (fixed) | Q3 |
| 9 | **Exp 3** — Layer-wise Sensitivity | Q4 |
| 10 | **Exp 4** — Feature Specificity | Q5 |
| 11 | **Exp 5** — Cross-Layer Transfer | Q6 |
| 12 | **Exp 6** — Epsilon Calibration | Q7 |
| 13 | Final Synthesis: Answers to Q1–Q7 | all |


## 1 · Smart Installer

Installs `sae-lens`, `transformer-lens`, `datasets`, and `transformers` once per
Colab session. A NumPy binary conflict between Colab's pre-installed NumPy and the
version required by `sae-lens` is resolved by calling `exit()`, which triggers
Colab's automatic runtime restart. After reconnection the imports succeed and
execution continues normally.

**Run this cell first, then click _Run All_ a second time to continue.**

The cell checks imports before calling `pip install` — if everything is already
present (e.g., on a second run) it exits immediately without re-installing.


In [ ]:
import os, sys

def _try_imports():
    try:
        import sae_lens, transformer_lens, datasets, transformers
        import numpy as np; np.random.seed(42)
        return True
    except (ImportError, ValueError):
        return False

if not _try_imports():
    print('Installing packages — runtime will restart automatically...')
    pkgs = (
        'sae-lens==4.3.1 '
        'transformer-lens==1.19.0 '
        'datasets==2.19.2 '
        'transformers==4.40.2 '
        'huggingface_hub '
        'tqdm matplotlib'
    )
    os.system('pip install -q ' + pkgs)
    print('Done. Restarting runtime...')
    exit()

print('All packages ready.')
import torch
print('PyTorch ' + torch.__version__ + ' | CUDA: ' + str(torch.cuda.is_available()))
if torch.cuda.is_available():
    os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')


## 2 · HuggingFace Token

**GPT-2:** leave `HF_TOKEN = ''` — no token needed.

**Gemma 2 2B:** paste your token (from `huggingface.co/settings/tokens`) into the
string in the next cell. Read-only access is sufficient.


In [ ]:
# ── Paste your HuggingFace token here (leave empty for GPT-2) ─────────────
HF_TOKEN = ''   # e.g. 'hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx'

# ── Login (runs only if token provided) ───────────────────────────────────
if HF_TOKEN.strip():
    from huggingface_hub import login as _hf_login
    _hf_login(token=HF_TOKEN.strip(), add_to_git_credential=False)
    print('Logged in to HuggingFace Hub.')
else:
    print('No token set — fine for GPT-2.')


## 3 · Model Configuration

**To switch between GPT-2 and Gemma 2 2B, change exactly one line:**

```python
MODEL_KEY = 'gpt2'    # GPT-2 Small   — no auth, ~500 MB, fast iteration
MODEL_KEY = 'gemma2'  # Gemma 2 2B    — HF auth required, ~10 GB VRAM on T4
```

All downstream cells read from the `CFG` dict — SAE release, hook template,
layer indices, `d_model` — so no other changes are needed anywhere in the
notebook.

All experiment hyperparameters (epsilon values, delta-KL values, number of
prompts, PGD steps) are defined in this cell and labelled clearly.


In [ ]:
import torch, json, math, time, os
import numpy as np
import torch.nn.functional as F
from tqdm.auto import tqdm
from pathlib import Path

torch.manual_seed(42)
np.random.seed(42)

# ╔══════════════════════════════════════════════════════════╗
# ║  MODEL SWITCH — change MODEL_KEY only, all else follows  ║
# ║  'gpt2'   → GPT-2 Small  (no auth, ~500 MB, fast)      ║
# ║  'gemma2' → Gemma 2 2B   (HF auth req, ~10 GB VRAM)    ║
# ╚══════════════════════════════════════════════════════════╝
MODEL_KEY = 'gpt2'   # ← ONLY LINE TO CHANGE

CONFIGS = {
    'gpt2': {
        'model_name':       'gpt2',
        'sae_release':      'gpt2-small-res-jb',
        'sae_id_tmpl':      'blocks.{layer}.hook_resid_pre',
        'hook_tmpl':        'blocks.{layer}.hook_resid_pre',
        'n_layers':         12,
        'target_layer':     8,
        'layers_to_test':   [0, 2, 4, 6, 8, 10, 11],
        'd_model':          768,
        'requires_hf_auth': False,
    },
    'gemma2': {
        'model_name':       'google/gemma-2-2b',
        'sae_release':      'gemma-scope-2b-pt-res',
        'sae_id_tmpl':      'layer_{layer}/width_16k/average_l0_82',
        'hook_tmpl':        'blocks.{layer}.hook_resid_pre',
        'n_layers':         26,
        'target_layer':     12,
        'layers_to_test':   [0, 4, 8, 12, 16, 20, 25],
        'd_model':          2304,
        'requires_hf_auth': True,
    },
}

CFG = CONFIGS[MODEL_KEY]

# ── Global hyperparameters ─────────────────────────────────────────────────
SEED            = 42
DEVICE          = 'cuda' if torch.cuda.is_available() else 'cpu'
N_BASELINE      = 150    # prompts for Exp 0
N_EXP           = 100    # prompts for Exp 1, 2
N_FAST          = 50     # prompts for Exp 3, 4, 5, 6

# Exp 1 — feature stability epsilon sweep (absolute L-inf values)
EPS_EXP1        = [0.01, 0.05, 0.1, 0.5, 1.0, 5.0]
PGD_STEPS       = 20

# Exp 2 — output-preserving attack (fixed from v2)
# Expressed as fractions of mean embedding norm (computed at runtime)
EPS_FRACS_EXP2  = [0.001, 0.005, 0.01, 0.05]
DELTA_KL_VALUES = [0.5, 1.0, 2.0, 5.0, 10.0]
OPA_STEPS       = 40     # more steps than PGD to give Lagrangian time to converge

# Exp 3 — layer-wise (two epsilon values for full picture)
EPS_EXP3        = [0.1, 1.0]

# Exp 5 — cross-layer transfer
EPS_EXP5        = 0.5
SRC_LAYERS      = [4, 8, 11] if MODEL_KEY == 'gpt2' else [8, 12, 20]

# Exp 6 — epsilon calibration relative to embedding norm
EPS_FRACS_EXP6  = [0.001, 0.005, 0.01, 0.05, 0.1, 0.2]

# Derived constants (do not change)
RESID_HOOK  = CFG['hook_tmpl'].format(layer=CFG['target_layer'])
EVAL_LAYERS = CFG['layers_to_test']

# ── Print config summary ───────────────────────────────────────────────────
sep = '=' * 60
print(sep)
print('  Config       : ' + MODEL_KEY)
print('  Model        : ' + CFG['model_name'])
print('  SAE release  : ' + CFG['sae_release'])
print('  Target hook  : ' + RESID_HOOK)
print('  Test layers  : ' + str(EVAL_LAYERS))
print('  Device       : ' + DEVICE)
print('  HF auth req  : ' + str(CFG['requires_hf_auth']))
print(sep)
print('  Eps  (Exp 1) : ' + str(EPS_EXP1))
print('  EpsFrac(Exp2): ' + str(EPS_FRACS_EXP2))
print('  DeltaKL(Exp2): ' + str(DELTA_KL_VALUES))
print('  SrcLayers(E5): ' + str(SRC_LAYERS))
print(sep)


## 4 · Drive Mount + Checkpoint System

Google Drive is mounted and three directories are created under
`SAE_Adversarial_Results_v3/`:

| Directory | Contents |
|-----------|---------|
| `results/` | Final merged JSON per experiment |
| `checkpoints/` | Per-(ε, layer, δ_KL) intermediate saves written *immediately* after each condition completes |
| `figures/` | Plots at 150 DPI |

The checkpoint system means a Colab disconnect never loses more than one
condition's worth of work. At the start of each experiment, completed
checkpoints are detected and skipped — the experiment resumes from where it
left off.

A write-access test is run at startup; the notebook raises an error
immediately rather than running for hours and then failing to save.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_BASE  = Path('/content/drive/MyDrive/SAE_Adversarial_Results_v3')
RESULTS_DIR = DRIVE_BASE / 'results'
CKPT_DIR    = DRIVE_BASE / 'checkpoints'
FIGS_DIR    = DRIVE_BASE / 'figures'

for _d in [RESULTS_DIR, CKPT_DIR, FIGS_DIR]:
    _d.mkdir(parents=True, exist_ok=True)

# Write-access test — fail loudly rather than run for hours then crash
_test = DRIVE_BASE / '.write_test'
try:
    _test.write_text('ok'); _test.unlink()
    print('Drive writable: ' + str(DRIVE_BASE))
except Exception as _e:
    raise RuntimeError('Drive not writable: ' + str(_e))

# ── Experiment status tracker ──────────────────────────────────────────────
EXP_STATUS   = {('exp' + str(i)): 'pending' for i in range(7)}
EXP_RUNTIMES = {}

def _mark(eid, status, t0=None):
    EXP_STATUS[eid] = status
    if t0 is not None:
        EXP_RUNTIMES[eid] = round(time.time() - t0, 1)
    icons = {'pending': '○', 'running': '◐', 'done': '●', 'error': '✗'}
    print('  [' + icons.get(status, '?') + '] ' + eid + ': ' + status)

# ── Checkpoint helpers ─────────────────────────────────────────────────────
def _ckpt_path(eid, key):
    safe = str(key).replace('/', '_').replace('.', 'p')
    return CKPT_DIR / (eid + '__' + safe + '.json')

def ckpt_exists(eid, key):
    return _ckpt_path(eid, key).exists()

def save_ckpt(eid, key, d):
    _ckpt_path(eid, key).write_text(json.dumps(d, indent=2, default=str))

def load_ckpt(eid, key):
    return json.loads(_ckpt_path(eid, key).read_text())

def save_result(fname, data):
    p = RESULTS_DIR / fname
    p.write_text(json.dumps(data, indent=2, default=str))
    print('  Saved -> ' + p.name)

def save_fig(fig, name):
    import matplotlib.pyplot as plt
    p = FIGS_DIR / name
    fig.savefig(p, dpi=150, bbox_inches='tight')
    plt.show()
    print('  Figure -> ' + p.name)

print('  results/    : ' + str(RESULTS_DIR))
print('  checkpoints/: ' + str(CKPT_DIR))
print('  figures/    : ' + str(FIGS_DIR))


## 5 · Model, SAE & Dataset Loading

The model is loaded via TransformerLens `HookedTransformer`. The SAE for
`target_layer` is loaded from `sae-lens`. The evaluation dataset is 150
WikiText-2 prompts (64–128 tokens, seed 42).

A **gradient-flow sanity check** runs automatically: it injects a test embedding
via `hook_embed`, reads the residual stream at `RESID_HOOK`, encodes through the
SAE, and calls `.backward()`. An assertion raises immediately if gradients
are `None` — the v1 bug that made all adversarial results invalid.

All **attack functions** are defined here and shared across every experiment:
- `pgd_attack` — PGD maximising SAE cosine disruption, logs per-step loss
- `output_preserving_attack` — Lagrangian PGD with KL constraint, logs λ and KL per step
- `random_baseline` — Monte-Carlo L-inf noise, used as comparison in every experiment

> **Gemma 2 note:** loading takes 3–5 min on T4 and uses ~9 GB VRAM.


In [ ]:
from transformer_lens import HookedTransformer
from sae_lens import SAE
from datasets import load_dataset
from transformers import AutoTokenizer

# ── Load model ─────────────────────────────────────────────────────────────
print('Loading model: ' + CFG['model_name'] + ' ...')
t0 = time.time()
model = HookedTransformer.from_pretrained(
    CFG['model_name'],
    center_unembed=True,
    center_writing_weights=True,
    fold_ln=True,
    device=DEVICE,
)
model.eval()
n_params = sum(p.numel() for p in model.parameters()) / 1e6
print('  Loaded in %.1fs | %.0fM params' % (time.time() - t0, n_params))

# ── Load SAE for target layer ───────────────────────────────────────────────
print('Loading SAE for layer %d ...' % CFG['target_layer'])
t0 = time.time()
sae_id = CFG['sae_id_tmpl'].format(layer=CFG['target_layer'])
sae, sae_cfg_dict, _ = SAE.from_pretrained(
    release=CFG['sae_release'],
    sae_id=sae_id,
    device=DEVICE,
)
sae.eval()
print('  Loaded in %.1fs | d_sae=%s | hook=%s' % (
    time.time() - t0,
    sae_cfg_dict.get('d_sae', '?'),
    sae_cfg_dict.get('hook_name', sae_id),
))

# ── Prepare dataset ─────────────────────────────────────────────────────────
print('Preparing evaluation dataset ...')
t0 = time.time()
_raw       = load_dataset('wikitext', 'wikitext-2-raw-v1', split='test')
_tokenizer = AutoTokenizer.from_pretrained(CFG['model_name'])
if _tokenizer.pad_token is None:
    _tokenizer.pad_token = _tokenizer.eos_token

_all_text = ' '.join([x for x in _raw['text'] if x.strip()])
_token_ids = _tokenizer.encode(_all_text)

np.random.seed(SEED)
_chunks, i = [], 0
while i < len(_token_ids) - 128 and len(_chunks) < N_BASELINE * 5:
    _l = np.random.randint(64, 129)
    c  = _token_ids[i : i + _l]
    if len(c) >= 64:
        _chunks.append(c)
    i += _l
np.random.shuffle(_chunks)
eval_set = _chunks[:N_BASELINE]

def to_tensor(tokens):
    return torch.tensor([tokens], dtype=torch.long, device=DEVICE)

print('  Dataset ready: %d prompts in %.1fs' % (len(eval_set), time.time() - t0))

# ── Mean embedding norm (needed by Exp 2 and Exp 6) ───────────────────────
with torch.no_grad():
    _norms = [model.embed(to_tensor(eval_set[i])).norm().item() for i in range(30)]
MEAN_EMBED_NORM = float(np.mean(_norms))
print('  Mean embed L2 norm: %.4f' % MEAN_EMBED_NORM)

# ── Gradient-flow sanity check ─────────────────────────────────────────────
print('Gradient-flow sanity check ...')
_toks  = to_tensor(eval_set[0])
_emb   = model.embed(_toks).detach().requires_grad_(True)
_store = {}
model.run_with_hooks(_toks, fwd_hooks=[
    ('hook_embed', lambda v, hook, e=_emb: e),
    (RESID_HOOK,   lambda v, hook, s=_store: (s.__setitem__('x', v), v)[1]),
])
sae.encode(_store['x']).sum().backward()
assert _emb.grad is not None, (
    'GRADIENT DID NOT FLOW — check TransformerLens version. '
    'Ensure run_with_hooks is used (not run_with_cache).'
)
print('  Gradient flow verified. Ready to run experiments.')


In [ ]:
# ── Shared attack functions (used by all experiments) ──────────────────────
#
# All functions use run_with_hooks (not run_with_cache) so the autograd graph
# is preserved. Hook callbacks use 'hook' as the second parameter name, which
# is required by TransformerLens: it calls hooks as fn(value, hook=hook_point).

def _get_acts(model, sae, tokens, resid_hook, embed_override=None):
    """Forward pass returning (logits, sae_acts). Always inside no_grad."""
    with torch.no_grad():
        store = {}
        hooks = [(resid_hook, lambda v, hook, s=store: (s.__setitem__('x', v), v)[1])]
        if embed_override is not None:
            _e = embed_override
            hooks = [('hook_embed', lambda v, hook, e=_e: e)] + hooks
        logits = model.run_with_hooks(tokens, fwd_hooks=hooks)
        acts   = sae.encode(store['x'])
    return logits, acts


def pgd_attack(model, sae, tokens, resid_hook, epsilon, steps=20, alpha=None):
    """PGD in embedding space maximising SAE cosine disruption.

    Returns (perturbed_embeds, info) where info contains loss history.
    Raises RuntimeError on step 0 if gradients do not flow.
    """
    if alpha is None:
        alpha = epsilon / 4.0
    tokens = tokens.to(DEVICE)

    with torch.no_grad():
        clean_emb  = model.embed(tokens)
    _, clean_acts = _get_acts(model, sae, tokens, resid_hook,
                              embed_override=clean_emb)
    clean_flat = clean_acts.detach().reshape(1, -1)

    delta     = torch.zeros_like(clean_emb)
    loss_hist = []

    for step in range(steps):
        perturbed = (clean_emb + delta).detach().requires_grad_(True)
        store = {}
        model.run_with_hooks(tokens, fwd_hooks=[
            ('hook_embed', lambda v, hook, p=perturbed: p),
            (resid_hook,   lambda v, hook, s=store:
             (s.__setitem__('x', v), v)[1]),
        ])
        acts_pert = sae.encode(store['x'])
        loss = -F.cosine_similarity(clean_flat,
                                    acts_pert.reshape(1, -1), dim=-1).mean()
        loss.backward()

        grad = perturbed.grad
        if step == 0 and grad is None:
            raise RuntimeError(
                'Gradient is None on step 0. run_with_hooks is breaking the '
                'autograd graph on this TransformerLens version.'
            )
        if grad is None:
            break
        loss_hist.append(float(loss.item()))
        with torch.no_grad():
            delta = (delta + alpha * grad.sign()).clamp(-epsilon, epsilon)
        perturbed.grad = None

    return (clean_emb + delta).detach(), {'loss_history': loss_hist}


def output_preserving_attack(model, sae, tokens, resid_hook, epsilon,
                              delta_kl=1.0, steps=40, alpha=None,
                              lambda_init=1.0, lambda_lr=0.1):
    """Lagrangian output-preserving PGD.

    Maximises SAE disruption subject to KL(p_clean || p_pert) <= delta_kl.
    Logs lambda and KL per step so convergence can be diagnosed.
    Raises RuntimeError on step 0 if gradients do not flow.
    """
    if alpha is None:
        alpha = epsilon / 4.0
    tokens = tokens.to(DEVICE)

    with torch.no_grad():
        clean_emb = model.embed(tokens)
    cl_logits, clean_acts = _get_acts(model, sae, tokens, resid_hook,
                                      embed_override=clean_emb)
    clean_flat = clean_acts.detach().reshape(1, -1)
    p_clean    = F.softmax(cl_logits[:, -1, :], dim=-1).detach()

    delta    = torch.zeros_like(clean_emb)
    lam      = float(lambda_init)
    lam_hist, kl_hist = [], []

    for step in range(steps):
        perturbed = (clean_emb + delta).detach().requires_grad_(True)
        store = {}
        pert_logits = model.run_with_hooks(tokens, fwd_hooks=[
            ('hook_embed', lambda v, hook, p=perturbed: p),
            (resid_hook,   lambda v, hook, s=store:
             (s.__setitem__('x', v), v)[1]),
        ])
        acts_pert = sae.encode(store['x'])
        sae_cos   = F.cosine_similarity(clean_flat,
                                        acts_pert.reshape(1, -1), dim=-1).mean()
        q_pert    = F.softmax(pert_logits[:, -1, :], dim=-1)
        kl        = F.kl_div(q_pert.log(), p_clean, reduction='batchmean')

        loss = -sae_cos + lam * torch.clamp(kl - delta_kl, min=0.0)
        loss.backward()

        grad = perturbed.grad
        if step == 0 and grad is None:
            raise RuntimeError('Gradient is None in output_preserving_attack.')
        if grad is None:
            break

        with torch.no_grad():
            kl_val  = float(kl.item())
            lam     = max(0.0, lam + lambda_lr * (kl_val - delta_kl))
            lam_hist.append(lam)
            kl_hist.append(kl_val)
            delta = (delta + alpha * grad.sign()).clamp(-epsilon, epsilon)
        perturbed.grad = None

    _, final_acts   = _get_acts(model, sae, tokens, resid_hook,
                                embed_override=(clean_emb + delta).detach())
    final_cos = float(F.cosine_similarity(clean_flat,
                                          final_acts.reshape(1, -1), dim=-1).mean())
    _, final_logits = _get_acts(model, sae, tokens, resid_hook,
                                embed_override=(clean_emb + delta).detach())
    q_final   = F.softmax(final_logits[:, -1, :], dim=-1)
    final_kl  = float(F.kl_div(q_final.log(), p_clean,
                                reduction='batchmean').item())

    return (clean_emb + delta).detach(), {
        'lambda_history': lam_hist,
        'kl_history':     kl_hist,
        'final_kl':       final_kl,
        'final_sae_cos':  final_cos,
        'lambda_final':   lam,
        'constraint_met': final_kl <= delta_kl,
    }


def random_baseline(model, sae, tokens, resid_hook, epsilon, n_samples=10):
    """Monte-Carlo L-inf noise baseline. Returns mean cosine, jaccard, flips."""
    tokens = tokens.to(DEVICE)
    with torch.no_grad():
        clean_emb  = model.embed(tokens)
        _, c_acts  = _get_acts(model, sae, tokens, resid_hook,
                               embed_override=clean_emb)
        c_flat     = c_acts.reshape(1, -1)
        cos_scores, jac_scores, flip_counts = [], [], []
        for _ in range(n_samples):
            noise  = torch.randn_like(clean_emb)
            linf   = noise.abs().max()
            noise  = noise / linf * epsilon if linf > 1e-12 else noise * 0
            _, p_a = _get_acts(model, sae, tokens, resid_hook,
                               embed_override=clean_emb + noise)
            cos = float(F.cosine_similarity(c_flat,
                                            p_a.reshape(1, -1), dim=-1).mean())
            a_b = c_acts.flatten() > 0
            b_b = p_a.flatten() > 0
            union = (a_b | b_b).sum().item()
            jac   = float((a_b & b_b).sum().item() / union) if union > 0 else 1.0
            cos_scores.append(cos)
            jac_scores.append(jac)
            flip_counts.append(float((a_b != b_b).sum().item()))
    return {
        'mean_cosine':     float(np.mean(cos_scores)),
        'std_cosine':      float(np.std(cos_scores)),
        'mean_jaccard':    float(np.mean(jac_scores)),
        'mean_flip_count': float(np.mean(flip_counts)),
    }


def metrics_from_acts(clean_acts, pert_acts, clean_logits=None, pert_logits=None):
    """Compute all metrics given clean and perturbed SAE activations."""
    c_f = clean_acts.flatten()
    p_f = pert_acts.flatten()
    cos = float(F.cosine_similarity(c_f.unsqueeze(0), p_f.unsqueeze(0)).item())
    a_b = c_f > 0
    b_b = p_f > 0
    union = (a_b | b_b).sum().item()
    jac   = float((a_b & b_b).sum().item() / union) if union > 0 else 1.0
    flips = float((a_b != b_b).sum().item())
    out   = {'cosine_similarity': cos, 'sae_disruption': 1.0 - cos,
             'jaccard_index': jac, 'feature_flip_count': flips}
    if clean_logits is not None and pert_logits is not None:
        p_c = F.softmax(clean_logits[:, -1, :], dim=-1)
        p_p = F.softmax(pert_logits[:, -1, :], dim=-1)
        out['kl_divergence'] = float(
            F.kl_div(p_p.log(), p_c, reduction='batchmean').item())
    return out

print('Attack functions defined.')


## 6 · Exp 0 — Baseline Stats

Before running any adversarial experiments, we establish clean baselines:
perplexity, next-token accuracy, SAE L0 (mean active features per token), and
SAE fraction of variance unexplained (FVU). These numbers serve as reference
points for interpreting how much adversarial perturbations change the model's
behaviour and the SAE's reconstruction quality.

Expected values for GPT-2 Small at layer 8: perplexity ≈ 58, accuracy ≈ 32%,
L0 ≈ 135, FVU ≈ 0.02.


In [ ]:
_mark('exp0', 'running')
t0_exp0 = time.time()

ppls, accs, l0s, fvus = [], [], [], []

for _toks_raw in tqdm(eval_set[:N_BASELINE], desc='Exp 0 baseline'):
    _toks = to_tensor(_toks_raw)
    with torch.no_grad():
        # Perplexity
        _logits = model(_toks)
        _log_probs = F.log_softmax(_logits, dim=-1)
        _targets   = _toks[:, 1:]
        _lp        = _log_probs[:, :-1, :].gather(
                         2, _targets.unsqueeze(-1)).squeeze(-1)
        ppls.append(float(torch.exp(-_lp.mean()).item()))

        # Next-token accuracy
        _preds   = _logits[:, :-1, :].argmax(-1)
        _correct = (_preds == _toks[:, 1:]).float().mean().item()
        accs.append(float(_correct))

        # SAE stats
        _, _acts = _get_acts(model, sae, _toks, RESID_HOOK)
        _acts_flat = _acts.reshape(-1, _acts.shape[-1])
        l0s.append(float((_acts_flat > 0).float().sum(-1).mean().item()))

        # FVU: var(residual) / var(input)
        _resid_store = {}
        model.run_with_hooks(_toks, fwd_hooks=[
            (RESID_HOOK, lambda v, hook, s=_resid_store:
             (s.__setitem__('x', v), v)[1])
        ])
        _x    = _resid_store['x'].reshape(-1, CFG['d_model'])
        _recon = sae.decode(sae.encode(_x))
        _fvu  = float((_x - _recon).var() / _x.var())
        fvus.append(_fvu)

baseline_results = {
    'experiment':    'baseline_v3',
    'model':         CFG['model_name'],
    'sae_release':   CFG['sae_release'],
    'target_layer':  CFG['target_layer'],
    'n_prompts':     N_BASELINE,
    'seed':          SEED,
    'perplexity':    {'mean': float(np.mean(ppls)), 'std': float(np.std(ppls))},
    'next_token_acc':{'mean': float(np.mean(accs)), 'std': float(np.std(accs))},
    'sae_l0':        {'mean': float(np.mean(l0s)),  'std': float(np.std(l0s))},
    'sae_fvu':       {'mean': float(np.mean(fvus)), 'std': float(np.std(fvus))},
    'mean_embed_norm': MEAN_EMBED_NORM,
}
save_result('exp0_baseline_v3.json', baseline_results)
_mark('exp0', 'done', t0_exp0)


In [ ]:
print('=== Exp 0 — Baseline Results ===')
print('  Perplexity     : %.2f ± %.2f' % (
    baseline_results['perplexity']['mean'],
    baseline_results['perplexity']['std']))
print('  Next-token acc : %.3f ± %.3f' % (
    baseline_results['next_token_acc']['mean'],
    baseline_results['next_token_acc']['std']))
print('  SAE L0         : %.1f ± %.1f' % (
    baseline_results['sae_l0']['mean'],
    baseline_results['sae_l0']['std']))
print('  SAE FVU        : %.4f ± %.4f' % (
    baseline_results['sae_fvu']['mean'],
    baseline_results['sae_fvu']['std']))
print('  Mean embed norm: %.4f' % MEAN_EMBED_NORM)
print('  Runtime        : %.1fs' % EXP_RUNTIMES.get('exp0', 0))


## 7 · Exp 1 — Feature Stability under PGD (Q1, Q2)

**Research questions:**
- **Q1:** Are SAE feature representations stable under adversarial embedding
  perturbations within a reasonable L-inf epsilon ball?
- **Q2:** At what epsilon does disruption become severe? Is there a threshold
  effect or is it gradual?

**Method:** PGD attack in embedding space across ε ∈ {0.01, 0.05, 0.1, 0.5,
1.0, 5.0}. For each ε, we measure adversarial SAE cosine similarity and compare
to a random-noise baseline at the same ε. The ratio (random cosine / adversarial
cosine) is the **adversarial advantage** — how many times more disruptive PGD is
than random perturbation of equal magnitude.

**Checkpoint strategy:** results are saved to Drive after each ε value
completes. If the runtime dies mid-sweep, re-running skips completed ε values
automatically.

**v2 finding:** peak advantage was 19,418× at ε = 0.05. Advantage collapses
to ~1.1× at ε ≥ 1.0 (random noise also saturates the SAE at large budgets).


In [ ]:
_mark('exp1', 'running')
t0_exp1 = time.time()
exp1_summary = {}
exp1_per_prompt = {}

for eps in EPS_EXP1:
    eps_key = str(eps)

    # ── Resume from checkpoint if available ───────────────────────────────
    if ckpt_exists('exp1', eps_key):
        print('  [skip] eps=%.3f — checkpoint found' % eps)
        _d = load_ckpt('exp1', eps_key)
        exp1_summary[eps_key]    = _d['summary']
        exp1_per_prompt[eps_key] = _d['per_prompt']
        continue

    print('  Running eps=%.3f ...' % eps)
    adv_rows, rnd_rows = [], []

    for _raw in tqdm(eval_set[:N_EXP], desc='  eps=%.3f' % eps, leave=False):
        _toks = to_tensor(_raw)

        # Adversarial
        _pert_emb, _info = pgd_attack(model, sae, _toks, RESID_HOOK,
                                       epsilon=eps, steps=PGD_STEPS)
        _cl_log, _cl_a = _get_acts(model, sae, _toks, RESID_HOOK)
        _pt_log, _pt_a = _get_acts(model, sae, _toks, RESID_HOOK,
                                    embed_override=_pert_emb)
        _m = metrics_from_acts(_cl_a, _pt_a, _cl_log, _pt_log)
        _m['loss_history'] = _info['loss_history']
        adv_rows.append(_m)

        # Random baseline
        _r = random_baseline(model, sae, _toks, RESID_HOOK, epsilon=eps)
        rnd_rows.append(_r)

    adv_cos  = np.mean([r['cosine_similarity'] for r in adv_rows])
    adv_std  = np.std( [r['cosine_similarity'] for r in adv_rows])
    rnd_cos  = np.mean([r['mean_cosine']        for r in rnd_rows])
    adv_kl   = np.mean([r.get('kl_divergence', float('nan')) for r in adv_rows])
    adv_flip = np.mean([r['feature_flip_count'] for r in adv_rows])
    advantage = float(rnd_cos / adv_cos) if adv_cos > 1e-9 else float('inf')

    summary = {
        'adv_cosine_mean': float(adv_cos), 'adv_cosine_std': float(adv_std),
        'rnd_cosine_mean': float(rnd_cos),
        'adv_kl_mean':     float(adv_kl),
        'adv_flip_mean':   float(adv_flip),
        'adv_advantage':   advantage,
    }
    exp1_summary[eps_key]    = summary
    exp1_per_prompt[eps_key] = {'adversarial': adv_rows, 'random': rnd_rows}

    save_ckpt('exp1', eps_key, {'summary': summary,
                                 'per_prompt': exp1_per_prompt[eps_key]})
    print('    adv_cos=%.4f | rnd_cos=%.4f | advantage=%.1fx | flips=%.0f' % (
          adv_cos, rnd_cos, advantage, adv_flip))

exp1_results = {
    'experiment': 'feature_stability_v3', 'model': CFG['model_name'],
    'target_layer': CFG['target_layer'], 'epsilon_values': EPS_EXP1,
    'pgd_steps': PGD_STEPS, 'n_prompts': N_EXP, 'seed': SEED,
    'summary': exp1_summary, 'per_prompt': exp1_per_prompt,
}
save_result('exp1_feature_stability_v3.json', exp1_results)
_mark('exp1', 'done', t0_exp1)


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

print('=== Exp 1 — Feature Stability Results ===')
print('%-8s  %-12s  %-12s  %-12s  %-12s' % (
      'eps', 'adv_cos', 'rnd_cos', 'advantage', 'flips'))
print('-' * 62)
for eps in EPS_EXP1:
    s = exp1_summary[str(eps)]
    print('%-8.3f  %-12.4f  %-12.4f  %-12.1f  %-12.0f' % (
          eps, s['adv_cosine_mean'], s['rnd_cosine_mean'],
          s['adv_advantage'], s['adv_flip_mean']))

# ── Plot: epsilon vs adversarial advantage (log scale) ────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

eps_vals  = [float(e) for e in EPS_EXP1]
adv_cos   = [exp1_summary[str(e)]['adv_cosine_mean'] for e in EPS_EXP1]
rnd_cos   = [exp1_summary[str(e)]['rnd_cosine_mean'] for e in EPS_EXP1]
advantage = [exp1_summary[str(e)]['adv_advantage']   for e in EPS_EXP1]

ax1.plot(eps_vals, adv_cos, 'o-', color='crimson',  label='Adversarial (PGD)')
ax1.plot(eps_vals, rnd_cos, 's--', color='steelblue', label='Random baseline')
ax1.axhline(0, color='gray', linewidth=0.8, linestyle=':')
ax1.set_xscale('log'); ax1.set_xlabel('epsilon (log scale)')
ax1.set_ylabel('SAE cosine similarity')
ax1.set_title('Cosine similarity vs epsilon')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(eps_vals, advantage, 'D-', color='darkorange')
ax2.axhline(1, color='gray', linewidth=0.8, linestyle=':', label='No advantage')
ax2.set_xscale('log'); ax2.set_yscale('log')
ax2.set_xlabel('epsilon (log scale)')
ax2.set_ylabel('Adversarial advantage (log scale)')
ax2.set_title('PGD advantage over random baseline')
ax2.grid(True, alpha=0.3); ax2.legend()

plt.tight_layout()
save_fig(fig, 'exp1_feature_stability.png')


## 8 · Exp 2 — Output-Preserving Attack, Fixed (Q3)

**Research question Q3:** Can an adversary disrupt SAE feature representations
while keeping the model's output distribution approximately unchanged?

**Why this was redesigned from v2:** In v2, the Lagrangian attack ran at ε = 0.5
with δ_KL ∈ {0.01, 0.1, 0.5, 1.0}. It failed completely — 0% constraint
satisfaction, output KL ≈ 18 nats regardless of target. Two explanations:
(a) **fundamental coupling**: in GPT-2's embedding space, any perturbation large
enough to disrupt SAEs also dramatically changes outputs; (b) **optimizer
failure**: the Lagrangian saddle point was never found because ε was too large.

**v3 fix:** test at much smaller ε (expressed as fractions of mean embedding
norm: 0.001–0.05) where SAE disruption is already highly adversarial (>2,000×
advantage per Exp 6) but the perturbation is subtler. δ_KL is widened to
{0.5, 1.0, 2.0, 5.0, 10.0} to find any feasible region.

**Key diagnostic:** per-step λ and KL trajectories are logged for every prompt,
enabling us to distinguish optimizer divergence from a genuine infeasibility
result.

**Checkpoint strategy:** one checkpoint per (ε_frac, δ_KL) pair, saved
immediately after each condition completes.


In [ ]:
_mark('exp2', 'running')
t0_exp2 = time.time()
exp2_grid = {}   # [eps_frac][delta_kl] -> summary + per_prompt

for eps_frac in EPS_FRACS_EXP2:
    eps_abs = eps_frac * MEAN_EMBED_NORM
    exp2_grid[str(eps_frac)] = {}

    for delta_kl in DELTA_KL_VALUES:
        ck_key = 'ef%s_dk%s' % (str(eps_frac), str(delta_kl))

        # ── Resume from checkpoint ─────────────────────────────────────────
        if ckpt_exists('exp2', ck_key):
            print('  [skip] eps_frac=%.3f delta_kl=%.1f' % (eps_frac, delta_kl))
            _d = load_ckpt('exp2', ck_key)
            exp2_grid[str(eps_frac)][str(delta_kl)] = _d
            continue

        print('  Running eps_frac=%.3f (abs=%.4f) delta_kl=%.1f ...' % (
              eps_frac, eps_abs, delta_kl))
        rows = []

        for _raw in tqdm(eval_set[:N_EXP], desc='  ef=%.3f dk=%.1f' % (
                         eps_frac, delta_kl), leave=False):
            _toks = to_tensor(_raw)
            try:
                _pert_emb, _info = output_preserving_attack(
                    model, sae, _toks, RESID_HOOK,
                    epsilon=eps_abs, delta_kl=delta_kl, steps=OPA_STEPS,
                )
                _cl_log, _cl_a = _get_acts(model, sae, _toks, RESID_HOOK)
                _pt_log, _pt_a = _get_acts(model, sae, _toks, RESID_HOOK,
                                            embed_override=_pert_emb)
                _m = metrics_from_acts(_cl_a, _pt_a, _cl_log, _pt_log)
                _m.update({
                    'final_kl':       _info['final_kl'],
                    'final_sae_cos':  _info['final_sae_cos'],
                    'lambda_final':   _info['lambda_final'],
                    'constraint_met': _info['constraint_met'],
                    # Store first and last 5 steps of trajectories (saves space)
                    'lambda_traj_head': _info['lambda_history'][:5],
                    'lambda_traj_tail': _info['lambda_history'][-5:],
                    'kl_traj_head':     _info['kl_history'][:5],
                    'kl_traj_tail':     _info['kl_history'][-5:],
                })
            except Exception as _e:
                _m = {'error': str(_e), 'constraint_met': False,
                      'cosine_similarity': float('nan'),
                      'sae_disruption': float('nan'),
                      'final_kl': float('nan')}
            rows.append(_m)

        n_ok  = sum(1 for r in rows if r.get('constraint_met', False))
        valid = [r for r in rows if not r.get('error')]
        summary = {
            'eps_frac':          eps_frac,
            'eps_abs':           eps_abs,
            'delta_kl_target':   delta_kl,
            'n_prompts':         len(rows),
            'n_succeeded':       n_ok,
            'pct_succeeded':     float(n_ok / len(rows) * 100),
            'mean_sae_disruption': float(np.nanmean(
                [r.get('sae_disruption', float('nan')) for r in valid])),
            'mean_output_kl':    float(np.nanmean(
                [r.get('final_kl', float('nan')) for r in valid])),
            'mean_lambda_final': float(np.nanmean(
                [r.get('lambda_final', float('nan')) for r in valid])),
        }
        _entry = {'summary': summary, 'per_prompt': rows}
        exp2_grid[str(eps_frac)][str(delta_kl)] = _entry
        save_ckpt('exp2', ck_key, _entry)

        print('    succeeded=%d/%d | sae_disrupt=%.3f | out_kl=%.2f | lam_fin=%.1f' % (
              n_ok, len(rows),
              summary['mean_sae_disruption'],
              summary['mean_output_kl'],
              summary['mean_lambda_final']))

exp2_results = {
    'experiment': 'output_preserving_v3', 'model': CFG['model_name'],
    'target_layer': CFG['target_layer'],
    'eps_fracs': EPS_FRACS_EXP2, 'delta_kl_values': DELTA_KL_VALUES,
    'mean_embed_norm': MEAN_EMBED_NORM, 'opa_steps': OPA_STEPS,
    'n_prompts': N_EXP, 'seed': SEED,
    'grid': exp2_grid,
}
save_result('exp2_output_preserving_v3.json', exp2_results)
_mark('exp2', 'done', t0_exp2)


In [ ]:
print('=== Exp 2 — Output-Preserving Attack Results ===')
print('%-8s  %-8s  %-8s  %-10s  %-10s  %-10s' % (
      'eps_frac', 'dKL_tgt', 'success%', 'sae_disrpt', 'out_kl', 'lam_fin'))
print('-' * 66)
for ef in EPS_FRACS_EXP2:
    for dk in DELTA_KL_VALUES:
        s = exp2_grid[str(ef)][str(dk)]['summary']
        print('%-8.3f  %-8.1f  %-8.1f  %-10.3f  %-10.2f  %-10.1f' % (
              ef, dk, s['pct_succeeded'],
              s['mean_sae_disruption'], s['mean_output_kl'],
              s['mean_lambda_final']))
    print()

# ── Plot: KL trajectory for one sample prompt at smallest eps_frac ─────────
_ef0 = str(EPS_FRACS_EXP2[0])
_dk0 = str(DELTA_KL_VALUES[0])
_sample_rows = [r for r in exp2_grid[_ef0][_dk0]['per_prompt']
                if 'kl_traj_head' in r]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

if _sample_rows:
    _r = _sample_rows[0]
    _kl_traj = _r['kl_traj_head'] + ['...'] + _r['kl_traj_tail']
    _kl_plot = _r['kl_traj_head'] + _r['kl_traj_tail']
    _lam_plot = _r['lambda_traj_head'] + _r['lambda_traj_tail']
    _x1 = list(range(len(_kl_plot)))
    _x2 = list(range(len(_lam_plot)))
    ax1.plot(_x1, _kl_plot, 'o-', color='crimson')
    ax1.axhline(float(DELTA_KL_VALUES[0]), color='green', linestyle='--',
                label='delta_kl target')
    ax1.set_xlabel('Step (head + tail)')
    ax1.set_ylabel('Output KL divergence (nats)')
    ax1.set_title('KL trajectory — eps_frac=%.3f, delta_kl=%.1f' % (
                  EPS_FRACS_EXP2[0], DELTA_KL_VALUES[0]))
    ax1.legend(); ax1.grid(True, alpha=0.3)
    ax2.plot(_x2, _lam_plot, 's-', color='darkorange')
    ax2.set_xlabel('Step (head + tail)')
    ax2.set_ylabel('Lambda (Lagrangian multiplier)')
    ax2.set_title('Lambda trajectory')
    ax2.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No trajectory data', ha='center', va='center',
             transform=ax1.transAxes)
    ax2.text(0.5, 0.5, 'No trajectory data', ha='center', va='center',
             transform=ax2.transAxes)

# Success rate heatmap
ax_heat_data = [[exp2_grid[str(ef)][str(dk)]['summary']['pct_succeeded']
                 for dk in DELTA_KL_VALUES] for ef in EPS_FRACS_EXP2]

plt.tight_layout()
save_fig(fig, 'exp2_trajectories.png')

# Second figure: success rate grid
fig2, ax = plt.subplots(figsize=(8, 4))
import numpy as _np2
_mat = _np2.array(ax_heat_data)
im = ax.imshow(_mat, aspect='auto', cmap='RdYlGn', vmin=0, vmax=100)
ax.set_xticks(range(len(DELTA_KL_VALUES)))
ax.set_xticklabels([str(d) for d in DELTA_KL_VALUES])
ax.set_yticks(range(len(EPS_FRACS_EXP2)))
ax.set_yticklabels([str(e) for e in EPS_FRACS_EXP2])
ax.set_xlabel('delta_kl target')
ax.set_ylabel('eps_frac')
ax.set_title('Exp 2 — Constraint satisfaction rate (%)')
plt.colorbar(im, ax=ax, label='% prompts with KL <= target')
for i in range(len(EPS_FRACS_EXP2)):
    for j in range(len(DELTA_KL_VALUES)):
        ax.text(j, i, '%.0f' % _mat[i, j], ha='center', va='center',
                color='black', fontsize=9)
plt.tight_layout()
save_fig(fig2, 'exp2_success_heatmap.png')


## 9 · Exp 3 — Layer-wise Sensitivity (Q4)

**Research question Q4:** Which transformer layers are most vulnerable to
adversarial SAE disruption? Does the pattern differ between moderate (ε = 0.1)
and large (ε = 1.0) budgets?

**Method:** For each layer in `EVAL_LAYERS`, load the layer-specific SAE, run
PGD at ε ∈ {0.1, 1.0}, and compute adversarial advantage vs random baseline.
The SAE is unloaded after each layer to avoid OOM, which is critical for Gemma 2.

**Checkpoint strategy:** one checkpoint per (ε, layer) pair. Layer-specific SAE
loading takes ~30–90s; checkpoints avoid reloading if the session resumes.

**v2 finding:** Layer 2 most vulnerable (916×) at ε = 0.1. Layer 4 follows
(7,288×). Layer 0 is least vulnerable (65×). At ε = 1.0 all layers collapse
to ~1.2× as random noise saturates the SAE.


In [ ]:
_mark('exp3', 'running')
t0_exp3 = time.time()
exp3_results_grid = {}   # [eps][layer] -> summary + per_prompt

for eps in EPS_EXP3:
    exp3_results_grid[str(eps)] = {}

    for layer in EVAL_LAYERS:
        ck_key = 'e%s_l%d' % (str(eps), layer)
        resid_hook_l = CFG['hook_tmpl'].format(layer=layer)

        # ── Resume from checkpoint ─────────────────────────────────────────
        if ckpt_exists('exp3', ck_key):
            print('  [skip] eps=%.2f layer=%d' % (eps, layer))
            _d = load_ckpt('exp3', ck_key)
            exp3_results_grid[str(eps)][str(layer)] = _d
            continue

        # ── Load layer-specific SAE ────────────────────────────────────────
        print('  Loading SAE layer=%d ...' % layer)
        _sae_id_l = CFG['sae_id_tmpl'].format(layer=layer)
        _sae_l, _, _ = SAE.from_pretrained(
            release=CFG['sae_release'], sae_id=_sae_id_l, device=DEVICE)
        _sae_l.eval()

        print('  Running eps=%.2f layer=%d ...' % (eps, layer))
        adv_rows, rnd_rows = [], []
        t_layer = time.time()

        for _raw in tqdm(eval_set[:N_FAST],
                         desc='  eps=%.2f L%d' % (eps, layer), leave=False):
            _toks = to_tensor(_raw)
            _pert_emb, _ = pgd_attack(model, _sae_l, _toks, resid_hook_l,
                                       epsilon=eps, steps=PGD_STEPS)
            _cl_log, _cl_a = _get_acts(model, _sae_l, _toks, resid_hook_l)
            _pt_log, _pt_a = _get_acts(model, _sae_l, _toks, resid_hook_l,
                                        embed_override=_pert_emb)
            adv_rows.append(metrics_from_acts(_cl_a, _pt_a, _cl_log, _pt_log))
            rnd_rows.append(random_baseline(model, _sae_l, _toks,
                                            resid_hook_l, epsilon=eps))

        # Unload SAE to free VRAM before next layer
        del _sae_l
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        adv_cos  = float(np.mean([r['cosine_similarity'] for r in adv_rows]))
        rnd_cos  = float(np.mean([r['mean_cosine']        for r in rnd_rows]))
        advantage = float(rnd_cos / adv_cos) if adv_cos > 1e-9 else float('inf')

        _entry = {
            'layer': layer, 'eps': eps,
            'summary': {
                'adv_cosine_mean': adv_cos,
                'adv_cosine_std':  float(np.std([r['cosine_similarity'] for r in adv_rows])),
                'rnd_cosine_mean': rnd_cos,
                'adv_flip_mean':   float(np.mean([r['feature_flip_count'] for r in adv_rows])),
                'adv_advantage':   advantage,
                'runtime_s':       time.time() - t_layer,
            },
            'per_prompt': {'adversarial': adv_rows, 'random': rnd_rows},
        }
        exp3_results_grid[str(eps)][str(layer)] = _entry
        save_ckpt('exp3', ck_key, _entry)
        print('    layer=%d | adv_cos=%.4f | rnd_cos=%.4f | advantage=%.1fx' % (
              layer, adv_cos, rnd_cos, advantage))

exp3_results = {
    'experiment': 'layerwise_v3', 'model': CFG['model_name'],
    'layers_tested': EVAL_LAYERS, 'eps_values': EPS_EXP3,
    'pgd_steps': PGD_STEPS, 'n_prompts': N_FAST, 'seed': SEED,
    'per_eps': exp3_results_grid,
}
save_result('exp3_layerwise_v3.json', exp3_results)
_mark('exp3', 'done', t0_exp3)


In [ ]:
print('=== Exp 3 — Layer-wise Sensitivity Results ===')
for eps in EPS_EXP3:
    print('\n  eps = %.2f' % eps)
    print('  %-6s  %-12s  %-12s  %-12s' % ('layer', 'adv_cos', 'rnd_cos', 'advantage'))
    print('  ' + '-' * 46)
    for layer in EVAL_LAYERS:
        s = exp3_results_grid[str(eps)][str(layer)]['summary']
        print('  %-6d  %-12.4f  %-12.4f  %-12.1f' % (
              layer, s['adv_cosine_mean'], s['rnd_cosine_mean'], s['adv_advantage']))

# ── Plot: bar chart of advantage per layer for each epsilon ────────────────
fig, axes = plt.subplots(1, len(EPS_EXP3), figsize=(6 * len(EPS_EXP3), 4),
                          sharey=False)
if len(EPS_EXP3) == 1:
    axes = [axes]

colors = ['#d62728', '#2196F3']
for ax, eps, col in zip(axes, EPS_EXP3, colors):
    advantages = [exp3_results_grid[str(eps)][str(l)]['summary']['adv_advantage']
                  for l in EVAL_LAYERS]
    ax.bar([str(l) for l in EVAL_LAYERS], advantages, color=col, alpha=0.8)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Adversarial advantage (x)')
    ax.set_title('Layer-wise advantage at eps=%.2f' % eps)
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3, axis='y')
    for i, (l, v) in enumerate(zip(EVAL_LAYERS, advantages)):
        ax.text(i, v * 1.05, '%.0fx' % v, ha='center', va='bottom', fontsize=8)

plt.tight_layout()
save_fig(fig, 'exp3_layerwise.png')


## 10 · Exp 4 — Feature Specificity (Q5)

**Research question Q5:** Does the PGD attack selectively suppress high-salience
features, or does it mainly create spurious new activations? What fraction of
the top-20 most important features (by activation magnitude) survive the attack?

**Method:** PGD at ε = 0.5, target layer. For each prompt, record:
- `n_suppressed`: features active in clean that become zero under attack
- `n_activated`: features zero in clean that become active under attack
- `n_stable_active`: features active in both
- `topk_survival_rate`: fraction of top-20 clean features still active after attack

**v2 finding:** The attack creates 4.7× more spurious activations (9,530) than
suppressions (2,030). Only 59.5% of top-20 features survive. The dominant effect
is hallucination of new features, not selective suppression — which has direct
implications for SAE-based interpretability monitoring systems.


In [ ]:
_mark('exp4', 'running')
t0_exp4 = time.time()
TOP_K   = 20
EPS_E4  = 0.5

if ckpt_exists('exp4', 'all'):
    print('  [skip] Exp 4 — checkpoint found')
    exp4_rows = load_ckpt('exp4', 'all')['per_prompt']
else:
    exp4_rows = []
    for _raw in tqdm(eval_set[:N_FAST], desc='Exp 4 feature specificity'):
        _toks = to_tensor(_raw)
        _pert_emb, _ = pgd_attack(model, sae, _toks, RESID_HOOK,
                                   epsilon=EPS_E4, steps=PGD_STEPS)
        with torch.no_grad():
            _, _cl_a = _get_acts(model, sae, _toks, RESID_HOOK)
            _, _pt_a = _get_acts(model, sae, _toks, RESID_HOOK,
                                  embed_override=_pert_emb)
            _c = _cl_a.flatten()
            _p = _pt_a.flatten()
            c_active = _c > 0
            p_active = _p > 0
            n_supp   = int((c_active & ~p_active).sum().item())
            n_act    = int(~c_active & p_active).sum().item()
            n_stable = int((c_active & p_active).sum().item())

            # Top-k survival rate
            _topk_idx = _c.topk(TOP_K).indices
            _topk_survived = p_active[_topk_idx].float().mean().item()

            # Suppression/activation ratios (normalised by total clean active)
            n_clean_total = int(c_active.sum().item())
            supp_ratio = float(n_supp / n_clean_total) if n_clean_total > 0 else 0.0
            act_ratio  = float(n_act  / n_clean_total) if n_clean_total > 0 else 0.0

            exp4_rows.append({
                'n_suppressed':    n_supp,
                'n_activated':     n_act,
                'n_stable_active': n_stable,
                'n_clean_active':  n_clean_total,
                'topk_survival_rate': float(_topk_survived),
                'suppression_ratio':  supp_ratio,
                'activation_ratio':   act_ratio,
                'mean_suppressed_act_clean': float(_c[c_active & ~p_active].mean().item())
                    if (c_active & ~p_active).any() else 0.0,
                'mean_activated_act_pert':   float(_p[~c_active & p_active].mean().item())
                    if (~c_active & p_active).any() else 0.0,
            })
    save_ckpt('exp4', 'all', {'per_prompt': exp4_rows})

def _mean(key): return float(np.mean([r[key] for r in exp4_rows]))
def _std(key):  return float(np.std( [r[key] for r in exp4_rows]))

exp4_results = {
    'experiment': 'feature_specificity_v3', 'model': CFG['model_name'],
    'target_layer': CFG['target_layer'], 'epsilon': EPS_E4,
    'top_k': TOP_K, 'n_prompts': N_FAST, 'seed': SEED,
    'summary': {k: {'mean': _mean(k), 'std': _std(k)} for k in [
        'n_suppressed', 'n_activated', 'n_stable_active',
        'topk_survival_rate', 'suppression_ratio', 'activation_ratio',
        'mean_suppressed_act_clean', 'mean_activated_act_pert',
    ]},
    'per_prompt': exp4_rows,
}
save_result('exp4_feature_specificity_v3.json', exp4_results)

print('=== Exp 4 — Feature Specificity Results ===')
for k in ['n_suppressed', 'n_activated', 'n_stable_active',
          'topk_survival_rate', 'suppression_ratio', 'activation_ratio']:
    print('  %-28s : %.2f ± %.2f' % (k, _mean(k), _std(k)))

# ── Plot: composition bar ──────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
cats   = ['Suppressed', 'Stable active', 'New activated']
means  = [_mean('n_suppressed'), _mean('n_stable_active'), _mean('n_activated')]
colors_bar = ['#e74c3c', '#27ae60', '#e67e22']
ax1.bar(cats, means, color=colors_bar, alpha=0.85)
ax1.set_ylabel('Mean feature count per prompt')
ax1.set_title('Feature composition under PGD (eps=%.1f)' % EPS_E4)
for i, v in enumerate(means):
    ax1.text(i, v + max(means)*0.01, '%.0f' % v, ha='center', fontsize=9)
ax1.grid(True, alpha=0.3, axis='y')

# Top-k survival rate distribution
topk_vals = [r['topk_survival_rate'] for r in exp4_rows]
ax2.hist(topk_vals, bins=20, color='steelblue', alpha=0.8, edgecolor='white')
ax2.axvline(np.mean(topk_vals), color='crimson', linestyle='--',
            label='mean=%.2f' % np.mean(topk_vals))
ax2.set_xlabel('Top-%d feature survival rate' % TOP_K)
ax2.set_ylabel('Count')
ax2.set_title('Top-k feature survival under PGD')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
save_fig(fig, 'exp4_feature_specificity.png')
_mark('exp4', 'done', t0_exp4)


## 11 · Exp 5 — Cross-Layer Transfer (Q6)

**Research question Q6:** Does a perturbation optimised to disrupt SAE features
at one layer also disrupt SAE features at other layers?

**Method:** PGD attack sourced from layers {4, 8, 11} (GPT-2) or {8, 12, 20}
(Gemma 2). Each attack is evaluated at all layers in `EVAL_LAYERS`. A separate
SAE is loaded per evaluation layer, used for evaluation only, then unloaded.

The result is a **cosine matrix** (src_layer × eval_layer) and an
**advantage matrix** (ratio of random baseline cosine to adversarial cosine at
each evaluation layer).

**Checkpoint strategy:** one checkpoint per (src_layer, eval_layer) pair.

**v2 finding:** Complete source-layer agnosticism — attacks from any source layer
produce almost identical disruption patterns across all evaluation layers.
Layer 0 resists cross-layer transfer (~1.65×). All other layers show 25–167×
advantage regardless of attack source.


In [ ]:
_mark('exp5', 'running')
t0_exp5 = time.time()
cosine_matrix    = {str(s): {} for s in SRC_LAYERS}
advantage_matrix = {str(s): {} for s in SRC_LAYERS}

for src_layer in SRC_LAYERS:
    src_hook = CFG['hook_tmpl'].format(layer=src_layer)
    src_sae_id = CFG['sae_id_tmpl'].format(layer=src_layer)

    # Load source SAE (for attack optimisation)
    print('  Loading src SAE layer=%d ...' % src_layer)
    _src_sae, _, _ = SAE.from_pretrained(
        release=CFG['sae_release'], sae_id=src_sae_id, device=DEVICE)
    _src_sae.eval()

    # Run PGD optimised for src_layer once per prompt
    print('  Running PGD for src_layer=%d ...' % src_layer)
    pert_embs = []
    clean_toks_list = []
    for _raw in tqdm(eval_set[:N_FAST], desc='  src=%d PGD' % src_layer,
                     leave=False):
        _toks = to_tensor(_raw)
        _pe, _ = pgd_attack(model, _src_sae, _toks, src_hook,
                             epsilon=EPS_EXP5, steps=PGD_STEPS)
        pert_embs.append(_pe)
        clean_toks_list.append(_toks)

    del _src_sae
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    # Evaluate disruption at every eval layer
    for eval_layer in EVAL_LAYERS:
        ck_key = 'src%d_eval%d' % (src_layer, eval_layer)

        if ckpt_exists('exp5', ck_key):
            print('  [skip] src=%d eval=%d' % (src_layer, eval_layer))
            _d = load_ckpt('exp5', ck_key)
            cosine_matrix[str(src_layer)][str(eval_layer)]    = _d['adv_cos']
            advantage_matrix[str(src_layer)][str(eval_layer)] = _d['advantage']
            continue

        eval_hook   = CFG['hook_tmpl'].format(layer=eval_layer)
        eval_sae_id = CFG['sae_id_tmpl'].format(layer=eval_layer)
        _eval_sae, _, _ = SAE.from_pretrained(
            release=CFG['sae_release'], sae_id=eval_sae_id, device=DEVICE)
        _eval_sae.eval()

        adv_cos_list, rnd_cos_list = [], []
        for _toks, _pe in zip(clean_toks_list, pert_embs):
            _, _cl_a = _get_acts(model, _eval_sae, _toks, eval_hook)
            _, _pt_a = _get_acts(model, _eval_sae, _toks, eval_hook,
                                  embed_override=_pe)
            adv_cos_list.append(
                float(F.cosine_similarity(_cl_a.reshape(1,-1),
                                          _pt_a.reshape(1,-1)).item()))
            _rnd = random_baseline(model, _eval_sae, _toks, eval_hook,
                                   epsilon=EPS_EXP5, n_samples=5)
            rnd_cos_list.append(_rnd['mean_cosine'])

        del _eval_sae
        if torch.cuda.is_available(): torch.cuda.empty_cache()

        adv_cos  = float(np.mean(adv_cos_list))
        rnd_cos  = float(np.mean(rnd_cos_list))
        advantage = float(rnd_cos / adv_cos) if adv_cos > 1e-9 else float('inf')
        cosine_matrix[str(src_layer)][str(eval_layer)]    = adv_cos
        advantage_matrix[str(src_layer)][str(eval_layer)] = advantage
        save_ckpt('exp5', ck_key, {'adv_cos': adv_cos, 'rnd_cos': rnd_cos,
                                    'advantage': advantage})
        print('    src=%d eval=%d | adv_cos=%.4f | advantage=%.1fx' % (
              src_layer, eval_layer, adv_cos, advantage))

exp5_results = {
    'experiment': 'cross_layer_transfer_v3', 'model': CFG['model_name'],
    'eps': EPS_EXP5, 'src_layers': SRC_LAYERS, 'eval_layers': EVAL_LAYERS,
    'pgd_steps': PGD_STEPS, 'n_prompts': N_FAST, 'seed': SEED,
    'cosine_matrix': cosine_matrix, 'advantage_matrix': advantage_matrix,
}
save_result('exp5_cross_layer_transfer_v3.json', exp5_results)

print('=== Exp 5 — Advantage Matrix (src x eval) ===')
header = '%-6s  ' % 'src\\eval' + '  '.join('%-6s' % str(l) for l in EVAL_LAYERS)
print(header)
for src in SRC_LAYERS:
    row = '%-6d  ' % src + '  '.join(
        '%-6.1f' % advantage_matrix[str(src)][str(l)] for l in EVAL_LAYERS)
    print(row)

# ── Heatmap of advantage matrix ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
import numpy as _np5
_mat5 = _np5.array([[advantage_matrix[str(s)][str(l)] for l in EVAL_LAYERS]
                     for s in SRC_LAYERS])
im = ax.imshow(_mat5, aspect='auto', cmap='YlOrRd')
ax.set_xticks(range(len(EVAL_LAYERS))); ax.set_xticklabels(EVAL_LAYERS)
ax.set_yticks(range(len(SRC_LAYERS)));  ax.set_yticklabels(SRC_LAYERS)
ax.set_xlabel('Evaluation layer'); ax.set_ylabel('Attack source layer')
ax.set_title('Exp 5 — Cross-layer transfer advantage (eps=%.1f)' % EPS_EXP5)
plt.colorbar(im, ax=ax, label='Adversarial advantage (x)')
for i in range(len(SRC_LAYERS)):
    for j in range(len(EVAL_LAYERS)):
        ax.text(j, i, '%.0f' % _mat5[i,j], ha='center', va='center',
                fontsize=8, color='black')
plt.tight_layout()
save_fig(fig, 'exp5_cross_layer_heatmap.png')
_mark('exp5', 'done', t0_exp5)


## 12 · Exp 6 — Epsilon Calibration (Q7)

**Research question Q7:** What is the minimum epsilon at which a targeted PGD
attack provides meaningful advantage over random noise? Is there a "safe" epsilon
below which SAE features are effectively robust?

**Method:** ε expressed as a fraction of the mean embedding L2 norm
(`MEAN_EMBED_NORM`, computed during dataset loading). Sweep over fractions
{0.001, 0.005, 0.01, 0.05, 0.1, 0.2}. This makes results model-agnostic:
ε_frac = 0.01 means the perturbation is 1% of the typical embedding magnitude,
regardless of whether the model is GPT-2 (norm ≈ 3.16) or Gemma 2 2B.

**Checkpoint strategy:** one checkpoint per ε_frac value.

**v2 finding:** Advantage is 21,036× even at ε_frac = 0.001 (perturbation =
0.1% of embedding norm, adversarial cosine = 0.997). No "safe" epsilon was
found across the tested range. Advantage decreases monotonically with ε as
random noise also becomes more disruptive at larger budgets.


In [ ]:
_mark('exp6', 'running')
t0_exp6 = time.time()
exp6_summary    = {}
exp6_per_prompt = {}

for eps_frac in EPS_FRACS_EXP6:
    eps_abs = eps_frac * MEAN_EMBED_NORM
    ck_key  = str(eps_frac)

    if ckpt_exists('exp6', ck_key):
        print('  [skip] eps_frac=%.3f' % eps_frac)
        _d = load_ckpt('exp6', ck_key)
        exp6_summary[ck_key]    = _d['summary']
        exp6_per_prompt[ck_key] = _d['per_prompt']
        continue

    print('  Running eps_frac=%.3f (abs=%.4f) ...' % (eps_frac, eps_abs))
    adv_rows, rnd_rows = [], []

    for _raw in tqdm(eval_set[:N_FAST], desc='  ef=%.3f' % eps_frac, leave=False):
        _toks = to_tensor(_raw)
        _pe, _info = pgd_attack(model, sae, _toks, RESID_HOOK,
                                 epsilon=eps_abs, steps=PGD_STEPS)
        _, _cl_a = _get_acts(model, sae, _toks, RESID_HOOK)
        _, _pt_a = _get_acts(model, sae, _toks, RESID_HOOK, embed_override=_pe)
        _m = metrics_from_acts(_cl_a, _pt_a)
        _m['loss_final'] = _info['loss_history'][-1] if _info['loss_history'] else None
        adv_rows.append(_m)
        rnd_rows.append(random_baseline(model, sae, _toks, RESID_HOOK,
                                        epsilon=eps_abs))

    adv_cos  = float(np.mean([r['cosine_similarity'] for r in adv_rows]))
    rnd_cos  = float(np.mean([r['mean_cosine']        for r in rnd_rows]))
    advantage = float(rnd_cos / adv_cos) if adv_cos > 1e-9 else float('inf')
    adv_flip  = float(np.mean([r['feature_flip_count'] for r in adv_rows]))

    summary = {
        'eps_frac': eps_frac, 'eps_abs': eps_abs,
        'adv_cosine_mean': adv_cos, 'rnd_cosine_mean': rnd_cos,
        'adv_advantage': advantage, 'adv_flip_mean': adv_flip,
    }
    exp6_summary[ck_key]    = summary
    exp6_per_prompt[ck_key] = {'adversarial': adv_rows, 'random': rnd_rows}
    save_ckpt('exp6', ck_key, {'summary': summary,
                                'per_prompt': exp6_per_prompt[ck_key]})
    print('    adv_cos=%.4f | rnd_cos=%.4f | advantage=%.1fx | flips=%.0f' % (
          adv_cos, rnd_cos, advantage, adv_flip))

exp6_results = {
    'experiment': 'epsilon_calibration_v3', 'model': CFG['model_name'],
    'target_layer': CFG['target_layer'], 'mean_embed_norm': MEAN_EMBED_NORM,
    'eps_fracs': EPS_FRACS_EXP6, 'pgd_steps': PGD_STEPS,
    'n_prompts': N_FAST, 'seed': SEED,
    'summary': exp6_summary, 'per_prompt': exp6_per_prompt,
}
save_result('exp6_epsilon_calibration_v3.json', exp6_results)

print('=== Exp 6 — Epsilon Calibration Results ===')
print('%-8s  %-8s  %-12s  %-12s  %-12s' % (
      'eps_frac', 'eps_abs', 'adv_cos', 'rnd_cos', 'advantage'))
print('-' * 58)
for ef in EPS_FRACS_EXP6:
    s = exp6_summary[str(ef)]
    print('%-8.3f  %-8.4f  %-12.4f  %-12.4f  %-12.1f' % (
          ef, s['eps_abs'], s['adv_cosine_mean'],
          s['rnd_cosine_mean'], s['adv_advantage']))

# ── Dual-axis plot ─────────────────────────────────────────────────────────
fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ef_vals   = [float(e) for e in EPS_FRACS_EXP6]
adv_cos_v = [exp6_summary[str(e)]['adv_cosine_mean'] for e in EPS_FRACS_EXP6]
adv_adv_v = [exp6_summary[str(e)]['adv_advantage']   for e in EPS_FRACS_EXP6]

ax1.plot(ef_vals, adv_cos_v, 'o-', color='steelblue', label='adv cosine sim')
ax1.set_xscale('log'); ax1.set_xlabel('eps_frac (log scale)')
ax1.set_ylabel('SAE cosine similarity', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

ax2.plot(ef_vals, adv_adv_v, 's--', color='crimson', label='advantage')
ax2.set_ylabel('Adversarial advantage (log scale)', color='crimson')
ax2.set_yscale('log')
ax2.tick_params(axis='y', labelcolor='crimson')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left')
ax1.set_title('Exp 6 — Epsilon calibration (norm-relative)')
ax1.grid(True, alpha=0.3)
plt.tight_layout()
save_fig(fig, 'exp6_epsilon_calibration.png')
_mark('exp6', 'done', t0_exp6)


## 13 · Final Synthesis — Answers to Q1–Q7

Loads all six result JSONs from Drive and produces a unified summary table
answering each research question with the key metric. This cell can be re-run
independently of the experiment cells, as long as the result files exist.


In [ ]:
def _load(fname):
    p = RESULTS_DIR / fname
    return json.loads(p.read_text()) if p.exists() else None

r0 = _load('exp0_baseline_v3.json')
r1 = _load('exp1_feature_stability_v3.json')
r2 = _load('exp2_output_preserving_v3.json')
r3 = _load('exp3_layerwise_v3.json')
r4 = _load('exp4_feature_specificity_v3.json')
r5 = _load('exp5_cross_layer_transfer_v3.json')
r6 = _load('exp6_epsilon_calibration_v3.json')

print('=' * 70)
print('  FINAL SYNTHESIS — SAE Adversarial Robustness v3')
print('  Model: ' + CFG['model_name'])
print('=' * 70)

# Q1 — feature stability
if r1:
    best_eps  = min(r1['summary'], key=lambda k: r1['summary'][k]['adv_cosine_mean'])
    best_adv  = r1['summary'][best_eps]['adv_advantage']
    print('\nQ1  Are SAE features stable under adversarial attack?')
    print('    NO. Peak advantage %.0fx at eps=%.3f. Features are highly fragile.' % (
          best_adv, float(best_eps)))

# Q2 — threshold
if r1:
    severe = [(k, v) for k, v in r1['summary'].items()
              if v['adv_cosine_mean'] < 0.1]
    print('\nQ2  At what eps does disruption become severe (cosine < 0.1)?')
    if severe:
        first_severe_eps = min(float(k) for k, _ in severe)
        print('    First severe at eps=%.3f.' % first_severe_eps)
    else:
        print('    Not reached in tested range — increase EPS_EXP1.')

# Q3 — output-preserving
if r2:
    total_ok = sum(
        r2['grid'][ef][dk]['summary']['n_succeeded']
        for ef in r2['grid'] for dk in r2['grid'][ef]
    )
    total_tried = sum(
        r2['grid'][ef][dk]['summary']['n_prompts']
        for ef in r2['grid'] for dk in r2['grid'][ef]
    )
    best_pct = max(
        r2['grid'][ef][dk]['summary']['pct_succeeded']
        for ef in r2['grid'] for dk in r2['grid'][ef]
    )
    print('\nQ3  Can output-preserving attacks disrupt SAEs?')
    print('    Total succeeded: %d / %d (%.1f%%). Best single condition: %.1f%%.' % (
          total_ok, total_tried, 100*total_ok/total_tried if total_tried else 0,
          best_pct))
    if best_pct == 0:
        print('    NEGATIVE: Lagrangian constraint never satisfied.')
    else:
        print('    POSITIVE: Constraint satisfied in some conditions.')

# Q4 — layer-wise
if r3:
    eps_str = str(EPS_EXP3[0])
    adv_by_layer = {
        l: r3['per_eps'][eps_str][str(l)]['summary']['adv_advantage']
        for l in EVAL_LAYERS if str(l) in r3['per_eps'].get(eps_str, {})
    }
    if adv_by_layer:
        most_vuln  = max(adv_by_layer, key=adv_by_layer.get)
        least_vuln = min(adv_by_layer, key=adv_by_layer.get)
        print('\nQ4  Which layers are most vulnerable (eps=%.2f)?' % EPS_EXP3[0])
        print('    Most vulnerable : layer %d (%.0fx)' % (
              most_vuln, adv_by_layer[most_vuln]))
        print('    Least vulnerable: layer %d (%.0fx)' % (
              least_vuln, adv_by_layer[least_vuln]))

# Q5 — feature specificity
if r4:
    s4 = r4['summary']
    print('\nQ5  Feature specificity of adversarial attacks?')
    print('    Suppressed: %.0f | New activated: %.0f | Stable: %.0f' % (
          s4['n_suppressed']['mean'], s4['n_activated']['mean'],
          s4['n_stable_active']['mean']))
    print('    Activation/suppression ratio: %.1fx | Top-k survival: %.1f%%' % (
          s4['activation_ratio']['mean'],
          s4['topk_survival_rate']['mean'] * 100))

# Q6 — cross-layer transfer
if r5:
    print('\nQ6  Do attacks transfer across layers?')
    src_0 = str(SRC_LAYERS[0])
    adv_at_non_src = [
        r5['advantage_matrix'][src_0][str(l)]
        for l in EVAL_LAYERS if l != SRC_LAYERS[0]
    ]
    print('    Attack from layer %s — mean advantage at other layers: %.1fx' % (
          src_0, float(np.mean(adv_at_non_src)) if adv_at_non_src else float('nan')))
    layer0_adv = r5['advantage_matrix'][src_0].get(str(EVAL_LAYERS[0]), None)
    if layer0_adv:
        print('    Layer 0 (embedding) resists transfer: %.1fx' % layer0_adv)

# Q7 — minimum epsilon
if r6:
    min_ef_s  = str(EPS_FRACS_EXP6[0])
    min_adv   = r6['summary'][min_ef_s]['adv_advantage']
    min_ef    = r6['summary'][min_ef_s]['eps_frac']
    print('\nQ7  Minimum epsilon for meaningful attack?')
    print('    At eps_frac=%.3f (norm-relative), advantage=%.0fx.' % (
          min_ef, min_adv))
    print('    No safe epsilon found in tested range.' if min_adv > 10
          else '    Low advantage at smallest eps — possible safe zone.')

print('\n' + '=' * 70)


## Runtime Summary & Experiment Status

Final status board — run this cell at any point to see which experiments have
completed and how long each took. The checkpoint system means completed
experiments will not re-run on the next execution even if this cell shows them
as "pending" (they were completed in a previous session and will be resumed via
checkpoints).


In [ ]:
print('=' * 50)
print('  Experiment Status Board')
print('=' * 50)
icons = {'pending': '○', 'running': '◐', 'done': '●', 'error': '✗'}
labels = {
    'exp0': 'Exp 0 — Baseline',
    'exp1': 'Exp 1 — Feature Stability (Q1, Q2)',
    'exp2': 'Exp 2 — Output-Preserving (Q3)',
    'exp3': 'Exp 3 — Layer-wise (Q4)',
    'exp4': 'Exp 4 — Feature Specificity (Q5)',
    'exp5': 'Exp 5 — Cross-Layer Transfer (Q6)',
    'exp6': 'Exp 6 — Epsilon Calibration (Q7)',
}
for eid, label in labels.items():
    status  = EXP_STATUS.get(eid, 'pending')
    runtime = EXP_RUNTIMES.get(eid)
    rt_str  = ('  [%.0fs]' % runtime) if runtime else ''
    print('  [%s] %s%s' % (icons.get(status, '?'), label, rt_str))

done_count = sum(1 for s in EXP_STATUS.values() if s == 'done')
print('=' * 50)
print('  Completed: %d / %d experiments' % (done_count, len(EXP_STATUS)))

# List all saved result files
print('\n  Results on Drive:')
for p in sorted(RESULTS_DIR.glob('*.json')):
    size_kb = p.stat().st_size / 1024
    print('    %-45s  %.1f KB' % (p.name, size_kb))

print('\n  Checkpoints on Drive:')
ckpts = list(CKPT_DIR.glob('*.json'))
print('    %d checkpoint files saved' % len(ckpts))
print('  Figures on Drive:')
figs = list(FIGS_DIR.glob('*.png'))
for f in sorted(figs):
    print('    ' + f.name)
